# Differential communication tutorial

This concise tutorial demonstrates one differential-CCC experiment. `A05` changes receiver-module activity while preserving sender choice; the same workflow applies to `N00`, `A01`-`A09`, `R01`-`R09`, and `M01`-`M09`. Prepared H5AD files are read directly, and the training call is commented by default.

> Start Jupyter from the repository root.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "model").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import anndata as ad
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from model.train.config import CellVOCALConfig
from pipeline.analysis import CellVOCALAnalysis
from pipeline.cellvocal import run_cellvocal


In [2]:
DATA_ROOT = ROOT / "paper" / "data" / "differential"
EXPERIMENT = "A05"

cfg = CellVOCALConfig()
cfg.train.device = "cuda:0" if torch.cuda.is_available() else "cpu"
cfg.train.checkpoint_dir = DATA_ROOT / "cellvocal" / EXPERIMENT / "checkpoints"
cfg.eval.output_dir = DATA_ROOT / "cellvocal" / EXPERIMENT / "results"
cfg.translator.mode = "off"
cfg.qc.normalize_per_spot = False
cfg.qc.n_top_genes = 500
cfg.model.use_cell_annotation = False
cfg.graph.n_spatial_neighbors = 24
cfg.graph.ccc_event_budget_per_receiver = 16
cfg.graph.ccc_ligand_expr_quantile = 0.15
cfg.graph.ccc_receptor_expr_quantile = 0.15
cfg.train.time_prior_source = "rna_maturity"
cfg.train.w_ccc = 0.8
cfg.train.w_time = 0.5

## Train the selected experiment

The simulator-provided LR table defines the admissible communication modules. It is a prior over candidate interactions, not a fitted result. Uncomment this cell when retraining is required.

In [3]:
# adata = ad.read_h5ad(DATA_ROOT / "raw" / f"{EXPERIMENT}.h5ad")
# lr = adata.uns["ground_truth"]["lr_pairs"]
# lr_pairs = [{"interaction": f"{lig}_{rec}", "pathway": str(mod), "module_id": str(mod), "ligand": str(lig), "receptor": str(rec), "ligand_genes": [str(lig)], "receptor_genes": [str(rec)]} for lig, rec, mod in zip(lr["ligand_gene"], lr["receptor_gene"], lr["module_id"])]
# result = run_cellvocal(adata, config=cfg, lr_pairs=lr_pairs)

## Inspect differential-effect recovery

`A05` perturbs receiver-module activity in 20% of the canonical groups without changing sender identity. The useful readout is therefore not a global average: it is the condition contrast within affected groups, shown beside untouched reference groups. Velocity maps are intentionally omitted here because they do not diagnose this perturbation.

In [4]:
analysis = CellVOCALAnalysis.from_h5ad(cfg.eval.output_dir / "out.h5ad", config=cfg)
design = analysis.adata.uns["ground_truth"]["differential"]

design_summary = pd.DataFrame({
    "value": [
        str(design["experiment_id"]),
        str(design["scenario"]),
        f'{int(design["n_affected_receiver_module_groups"]):,} / {int(design["n_receiver_module_groups"]):,}',
        f'{100 * float(design["realized_affected_fraction"]):.1f}%',
        float(design["effect_strength"]),
        f'{100 * float(design["affected_top_sender_switch_rate"]):.1f}%',
    ]
}, index=["experiment", "perturbation", "affected receiver-module groups", "realized affected fraction", "effect strength", "top-sender switch rate"])
display(design_summary)

ccc = analysis.adata.uns["cv_ccc"]
affected_lookup = {
    (int(receiver), str(module))
    for receiver, module, affected in zip(
        design["group_receiver_id"], design["group_module_id"], design["group_is_affected"]
    )
    if affected
}
receiver = np.asarray(ccc["receiver"], dtype=int)
canonical_receiver = analysis.adata.obs.iloc[receiver]["canonical_cell_id"].astype(int).to_numpy()
module_id = np.asarray(ccc["lr_pair_module_ids"], dtype=str)[np.asarray(ccc["module"], dtype=int)]
condition = analysis.adata.obs.iloc[receiver]["condition"].astype(str).to_numpy()
group = np.where(
    [(int(r), str(m)) in affected_lookup for r, m in zip(canonical_receiver, module_id)],
    "affected", "reference"
)
event_summary = (
    pd.DataFrame({"group": group, "condition": condition, "event_influence": np.asarray(ccc["event_influence"], dtype=float)})
    .groupby(["group", "condition"])["event_influence"]
    .agg(candidate_events="size", mean_event_influence="mean")
)
contrast = pd.DataFrame(index=["affected", "reference"])
contrast["control events"] = event_summary["candidate_events"].unstack()["control"]
contrast["perturbed events"] = event_summary["candidate_events"].unstack()["perturbed"]
contrast["control mean"] = event_summary["mean_event_influence"].unstack()["control"]
contrast["perturbed mean"] = event_summary["mean_event_influence"].unstack()["perturbed"]
contrast["change vs control (%)"] = 100 * (contrast["perturbed mean"] / contrast["control mean"] - 1)
display(contrast.round({"control mean": 4, "perturbed mean": 4, "change vs control (%)": 1}))

,value
experiment,A05
perturbation,activity
affected receiver-module groups,96 / 480
realized affected fraction,20.0%
effect strength,0.5
top-sender switch rate,0.0%


,control events,perturbed events,control mean,perturbed mean,change vs control (%)
affected,4608,4608,0.0645,0.0728,12.9
reference,32604,32071,0.0461,0.0437,-5.1


For the full benchmark, repeat the same independent fit for each experiment ID and compare conditions only after every run uses the same model configuration.